# DS4/GSE278450 — memory-safe construction of the merged raw-count AnnData object

This notebook reconstructs the GSE278450 object from the 22 compressed
sample-level files ending in `_RawData.tsv.gz`.

The input files are gene-by-cell tables. To keep peak RAM bounded on a
16-GB workstation, this version:

- parses each gzip TSV in small gene-row chunks;
- reproduces the original pandas parser semantics in small chunks;
- detects the GEO layout where the header contains only cell barcodes and each data row has one additional, unlabeled gene-identifier field;
- converts each chunk directly to sparse CSR;
- writes one temporary H5AD per sample and releases it from memory;
- merges the sample files with `anndata.experimental.concat_on_disk`;
- restores dataset-level `uns` metadata through a small sidecar H5AD;
- validates the final sparse matrix directly through HDF5 without loading it.

Successful sample caches are reused after an interrupted run. They are deleted
only after the final output has passed validation.

## Expected merged object

- 22 samples
- 91,336 cells
- 36,116 genes

## Default paths

Input:

`data/raw/GSE278450/`

Output:

`data/interim/DS4_GSE278450/GSE278450_merged_rawcounts.h5ad`

Paths can be overridden with `GLIOMA_PROJECT_ROOT`,
`GSE278450_RAW_DIR`, and `GSE278450_MERGED_H5AD`.

The parser chunk size can be reduced further with
`GSE278450_GENE_CHUNK_ROWS` if required.


In [1]:
import csv
import gc
import gzip
import inspect
import json
import os
import re
import shutil
import sys
from copy import deepcopy
from datetime import datetime
from pathlib import Path

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scanpy as sc
import scipy
import scipy.sparse as sp

try:
    from anndata.experimental import concat_on_disk
except ImportError as exc:
    raise ImportError(
        "This notebook requires anndata.experimental.concat_on_disk. "
        "Use the glioma-cnv-atlas environment that was used for DS3."
    ) from exc

try:
    from anndata.io import read_elem
except ImportError:
    # Compatibility with older AnnData releases.
    from anndata._io.specs import read_elem

print("Python:", sys.version)
print("anndata:", ad.__version__)
print("scanpy:", sc.__version__)
print("pandas:", pd.__version__)
print("scipy:", scipy.__version__)
print("concat_on_disk:", inspect.signature(concat_on_disk))


Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
anndata: 0.12.7
scanpy: 1.11.5
pandas: 2.3.3
scipy: 1.16.3
concat_on_disk: (in_files: 'Collection[PathLike[str] | str | H5Group | ZarrGroup] | Mapping[str, PathLike[str] | str | H5Group | ZarrGroup]', out_file: 'PathLike[str] | str | H5Group | ZarrGroup', *, max_loaded_elems: 'int' = 100000000, axis: "Literal['obs', 0, 'var', 1]" = 0, join: "Literal['inner', 'outer']" = 'inner', merge: 'StrategiesLiteral | Callable[[Collection[Mapping]], Mapping] | None' = None, uns_merge: 'StrategiesLiteral | Callable[[Collection[Mapping]], Mapping] | None' = None, label: 'str | None' = None, keys: 'Collection[str] | None' = None, index_unique: 'str | None' = None, fill_value: 'Any | None' = None, pairwise: 'bool' = False) -> 'None'


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_27700\2094694179.py:37: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)
C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_27700\2094694179.py:38: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)


In [2]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

RAW_DIR = Path(
    os.environ.get(
        "GSE278450_RAW_DIR",
        PROJECT_ROOT / "data" / "raw" / "GSE278450",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE278450_MERGED_H5AD",
        PROJECT_ROOT
        / "data"
        / "interim"
        / "DS4_GSE278450"
        / "GSE278450_merged_rawcounts.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT / "results" / "qc" / "DS4_GSE278450" / "00a_build"
)

SAMPLE_CACHE_DIR = (
    OUTPUT_H5AD.parent / ".GSE278450_00a_sample_cache_v5"
)

EXPECTED_SAMPLE_IDS = {
    "GSM8546833_BT356",
    "GSM8546834_BT363",
    "GSM8546835_BT364",
    "GSM8546836_BT368",
    "GSM8546837_BT369",
    "GSM8546838_BT373",
    "GSM8546839_BT374",
    "GSM8546840_BT377",
    "GSM8546841_BT380",
    "GSM8546842_BT389A",
    "GSM8546843_BT389",
    "GSM8546844_BT390",
    "GSM8546845_BT396",
    "GSM8546846_BT397",
    "GSM8546847_BT400",
    "GSM8546848_BT402",
    "GSM8546849_BT407",
    "GSM8546850_BT408",
    "GSM8546851_BT409",
    "GSM8546852_BT419",
    "GSM8546853_BT420",
    "GSM8546854_BT426",
}

EXPECTED_N_SAMPLES = 22
EXPECTED_OUTPUT_SHAPE = (91_336, 36_116)

GENE_CHUNK_ROWS = int(
    os.environ.get("GSE278450_GENE_CHUNK_ROWS", "128")
)
CONCAT_MAX_LOADED_ELEMS = int(
    os.environ.get("GSE278450_CONCAT_MAX_LOADED_ELEMS", "5000000")
)
VALIDATION_DATA_CHUNK = int(
    os.environ.get("GSE278450_VALIDATION_DATA_CHUNK", "2000000")
)
MINIMUM_INITIAL_FREE_GIB = float(
    os.environ.get("GSE278450_MINIMUM_FREE_GIB", "50")
)

CACHE_VERSION = "ds4-00a-memory-safe-v5"

if GENE_CHUNK_ROWS < 1:
    raise ValueError("GSE278450_GENE_CHUNK_ROWS must be positive.")
if CONCAT_MAX_LOADED_ELEMS < 1:
    raise ValueError("GSE278450_CONCAT_MAX_LOADED_ELEMS must be positive.")
if VALIDATION_DATA_CHUNK < 1:
    raise ValueError("GSE278450_VALIDATION_DATA_CHUNK must be positive.")

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("RAW directory:", RAW_DIR)
print("Output:", OUTPUT_H5AD)
print("Sample cache:", SAMPLE_CACHE_DIR)
print("Gene rows per parser chunk:", GENE_CHUNK_ROWS)
print("concat_on_disk max_loaded_elems:", CONCAT_MAX_LOADED_ELEMS)

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"GSE278450 RAW directory not found: {RAW_DIR}\n"
        "Extract the *_RawData.tsv.gz files there or set "
        "GSE278450_RAW_DIR."
    )

initial_free_bytes = shutil.disk_usage(OUTPUT_H5AD.parent).free
print(
    "Initial free disk space:",
    f"{initial_free_bytes / 1024**3:.2f} GiB",
)

if initial_free_bytes < int(MINIMUM_INITIAL_FREE_GIB * 1024**3):
    raise OSError(
        "Insufficient free disk space for sample caches and the final "
        f"on-disk merge. At least {MINIMUM_INITIAL_FREE_GIB:.0f} GiB "
        f"is required; observed {initial_free_bytes / 1024**3:.2f} GiB."
    )


Project root: G:\Repository\glioma-cnv-single-cell-atlas
RAW directory: G:\Repository\glioma-cnv-single-cell-atlas\data\raw\GSE278450
Output: G:\Repository\glioma-cnv-single-cell-atlas\data\interim\DS4_GSE278450\GSE278450_merged_rawcounts.h5ad
Sample cache: G:\Repository\glioma-cnv-single-cell-atlas\data\interim\DS4_GSE278450\.GSE278450_00a_sample_cache_v5
Gene rows per parser chunk: 128
concat_on_disk max_loaded_elems: 5000000
Initial free disk space: 827.95 GiB


In [3]:
FILE_PATTERN = re.compile(r"^(GSM\d+_.+)_RawData\.tsv\.gz$")


def discover_raw_tables(raw_dir: Path) -> list[dict]:
    records = []

    for path in sorted(raw_dir.glob("*_RawData.tsv.gz")):
        match = FILE_PATTERN.match(path.name)
        if match is None:
            raise ValueError(
                f"Unexpected RawData filename: {path.name!r}"
            )

        records.append(
            {
                "sample_id": match.group(1),
                "path": path,
            }
        )

    observed = {record["sample_id"] for record in records}

    if observed != EXPECTED_SAMPLE_IDS:
        missing = sorted(EXPECTED_SAMPLE_IDS - observed)
        extra = sorted(observed - EXPECTED_SAMPLE_IDS)
        raise ValueError(
            "The discovered GSE278450 RawData files do not match the "
            f"historical 22-sample set. Missing={missing}; extra={extra}"
        )

    if len(records) != EXPECTED_N_SAMPLES:
        raise AssertionError(
            f"Expected {EXPECTED_N_SAMPLES} RawData files, "
            f"observed {len(records)}."
        )

    return records


input_records = discover_raw_tables(RAW_DIR)

inventory = pd.DataFrame(
    [
        {
            "sample_id": record["sample_id"],
            "filename": record["path"].name,
            "compressed_size_bytes": record["path"].stat().st_size,
            "modified_time_ns": record["path"].stat().st_mtime_ns,
        }
        for record in input_records
    ]
)

inventory.to_csv(
    AUDIT_DIR / "GSE278450_input_file_inventory.tsv",
    sep="\t",
    index=False,
)

print(inventory)
print(
    "Total compressed input size:",
    f"{inventory['compressed_size_bytes'].sum() / 1024**3:.2f} GiB",
)


            sample_id                          filename  \
0    GSM8546833_BT356   GSM8546833_BT356_RawData.tsv.gz   
1    GSM8546834_BT363   GSM8546834_BT363_RawData.tsv.gz   
2    GSM8546835_BT364   GSM8546835_BT364_RawData.tsv.gz   
3    GSM8546836_BT368   GSM8546836_BT368_RawData.tsv.gz   
4    GSM8546837_BT369   GSM8546837_BT369_RawData.tsv.gz   
5    GSM8546838_BT373   GSM8546838_BT373_RawData.tsv.gz   
6    GSM8546839_BT374   GSM8546839_BT374_RawData.tsv.gz   
7    GSM8546840_BT377   GSM8546840_BT377_RawData.tsv.gz   
8    GSM8546841_BT380   GSM8546841_BT380_RawData.tsv.gz   
9   GSM8546842_BT389A  GSM8546842_BT389A_RawData.tsv.gz   
10   GSM8546843_BT389   GSM8546843_BT389_RawData.tsv.gz   
11   GSM8546844_BT390   GSM8546844_BT390_RawData.tsv.gz   
12   GSM8546845_BT396   GSM8546845_BT396_RawData.tsv.gz   
13   GSM8546846_BT397   GSM8546846_BT397_RawData.tsv.gz   
14   GSM8546847_BT400   GSM8546847_BT400_RawData.tsv.gz   
15   GSM8546848_BT402   GSM8546848_BT402_RawData.tsv.gz 

In [4]:
def decode_h5_attr(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return value


def stored_matrix_shape(node) -> tuple[int, int]:
    if isinstance(node, h5py.Dataset):
        return tuple(int(value) for value in node.shape)

    if not isinstance(node, h5py.Group):
        raise TypeError(
            f"Unsupported HDF5 matrix node: {type(node).__name__}"
        )

    shape = node.attrs.get("shape")
    if shape is None:
        raise KeyError(
            f"Sparse matrix group {node.name!r} has no shape attribute."
        )

    return tuple(int(value) for value in shape)


def stored_sparse_encoding(group: h5py.Group) -> str:
    return str(decode_h5_attr(group.attrs.get("encoding-type", "")))


def inspect_raw_tsv_layout(path: Path) -> dict:
    with gzip.open(
        path,
        mode="rt",
        encoding="utf-8-sig",
        newline="",
    ) as handle:
        reader = csv.reader(handle, delimiter="\t")

        try:
            header = next(reader)
        except StopIteration as exc:
            raise ValueError(f"{path.name}: the gzip TSV is empty.") from exc

        first_data_row = None
        for row in reader:
            if row and any(str(value).strip() for value in row):
                first_data_row = row
                break

    if first_data_row is None:
        raise ValueError(
            f"{path.name}: no non-empty data row follows the header."
        )

    header_fields = int(len(header))
    first_data_fields = int(len(first_data_row))

    if first_data_fields == header_fields + 1:
        layout = "cell-only header with unlabeled gene-index field"
    elif first_data_fields == header_fields:
        layout = "labeled first column"
    else:
        raise ValueError(
            f"{path.name}: unsupported TSV width relationship; "
            f"header has {header_fields} fields but the first data row "
            f"has {first_data_fields} fields."
        )

    return {
        "layout": layout,
        "header_field_count": header_fields,
        "first_data_field_count": first_data_fields,
        "raw_header_first_field": (
            str(header[0]) if header else ""
        ),
    }


def probe_historical_pandas_parser(
    path: Path,
) -> tuple[pd.Index, dict]:
    layout = inspect_raw_tsv_layout(path)

    # A one-row probe is memory-safe and reproduces exactly how the
    # historical pd.read_csv(..., index_col=0, low_memory=False) call
    # interprets a header that is one field shorter than the data rows.
    probe = pd.read_csv(
        path,
        sep="\t",
        compression="gzip",
        index_col=0,
        low_memory=False,
        nrows=1,
    )

    if probe.shape[0] != 1 or probe.shape[1] == 0:
        raise ValueError(
            f"{path.name}: the historical parser probe is empty."
        )

    cell_barcodes = pd.Index(
        probe.columns.astype(str),
        name="cell_barcode",
        dtype="object",
    )

    if not cell_barcodes.is_unique:
        examples = (
            cell_barcodes[cell_barcodes.duplicated(keep=False)]
            .unique()
            .tolist()[:10]
        )
        raise ValueError(
            f"{path.name}: duplicate parsed cell columns were found, "
            f"including {examples}."
        )

    layout["parsed_cell_count"] = int(len(cell_barcodes))
    layout["parsed_index_name"] = (
        None if probe.index.name is None else str(probe.index.name)
    )
    layout["first_parsed_gene"] = str(probe.index[0])

    return cell_barcodes, layout


def validate_dense_count_chunk(
    values: np.ndarray,
    source: Path,
    chunk_number: int,
) -> tuple[np.ndarray, int, int]:
    if values.ndim != 2 or values.size == 0:
        raise ValueError(
            f"{source.name}: parser chunk {chunk_number} is empty."
        )

    if not np.isfinite(values).all():
        raise ValueError(
            f"{source.name}: non-finite count in chunk {chunk_number}."
        )
    if np.any(values < 0):
        raise ValueError(
            f"{source.name}: negative count in chunk {chunk_number}."
        )

    rounded = np.rint(values)
    if not np.array_equal(values, rounded):
        raise ValueError(
            f"{source.name}: non-integer count in chunk {chunk_number}."
        )

    chunk_maximum = int(values.max())
    if chunk_maximum > np.iinfo(np.int32).max:
        raise OverflowError(
            f"{source.name}: a count exceeds int32 in chunk "
            f"{chunk_number}: {chunk_maximum}"
        )

    integer_values = values.astype(np.int32, copy=False)
    chunk_total = int(integer_values.sum(dtype=np.int64))

    return integer_values, chunk_total, chunk_maximum


def validate_cached_sample(
    cache_path: Path,
    source_path: Path,
    sample_id: str,
) -> tuple[dict, pd.Index]:
    if not cache_path.exists():
        raise FileNotFoundError(cache_path)

    source_stat = source_path.stat()

    with h5py.File(cache_path, "r") as handle:
        for key in ("X", "obs", "var", "uns"):
            if key not in handle:
                raise AssertionError(
                    f"{cache_path.name}: missing H5AD group /{key}."
                )

        if not isinstance(handle["X"], h5py.Group):
            raise AssertionError(
                f"{cache_path.name}: cached X is unexpectedly dense."
            )

        encoding = stored_sparse_encoding(handle["X"])
        if encoding != "csr_matrix":
            raise AssertionError(
                f"{cache_path.name}: expected CSR X, observed "
                f"{encoding!r}."
            )

        if handle["X"]["data"].dtype != np.dtype(np.int32):
            raise AssertionError(
                f"{cache_path.name}: expected int32 X data, observed "
                f"{handle['X']['data'].dtype}."
            )

        shape = stored_matrix_shape(handle["X"])
        stored_nnz = int(handle["X"]["data"].shape[0])
        obs = read_elem(handle["obs"])
        var = read_elem(handle["var"])
        uns = read_elem(handle["uns"])

    if shape != (len(obs), len(var)):
        raise AssertionError(
            f"{cache_path.name}: matrix/metadata shape mismatch."
        )
    if not obs.index.is_unique:
        raise AssertionError(
            f"{cache_path.name}: cached cell identifiers are not unique."
        )
    if not var.index.is_unique:
        raise AssertionError(
            f"{cache_path.name}: cached gene identifiers are not unique."
        )
    if "sample_id" not in obs.columns:
        raise AssertionError(
            f"{cache_path.name}: missing obs['sample_id']."
        )

    observed_samples = set(obs["sample_id"].astype(str).unique())
    if observed_samples != {sample_id}:
        raise AssertionError(
            f"{cache_path.name}: unexpected sample IDs "
            f"{sorted(observed_samples)}."
        )

    required_uns = {
        "cache_version": CACHE_VERSION,
        "source_file": source_path.name,
        "source_file_size_bytes": int(source_stat.st_size),
        "source_file_mtime_ns": int(source_stat.st_mtime_ns),
        "sample_id": sample_id,
    }

    for key, expected in required_uns.items():
        if key not in uns:
            raise AssertionError(
                f"{cache_path.name}: missing uns[{key!r}]."
            )
        observed = uns[key]
        if isinstance(observed, np.generic):
            observed = observed.item()
        if observed != expected:
            raise AssertionError(
                f"{cache_path.name}: uns[{key!r}] mismatch; "
                f"expected {expected!r}, observed {observed!r}."
            )

    summary = {
        "sample_id": sample_id,
        "cells": int(shape[0]),
        "genes": int(shape[1]),
        "total_counts": int(uns["total_counts"]),
        "maximum_count": int(uns["maximum_count"]),
        "nonzero_entries": int(uns["nonzero_entries"]),
        "source_file": source_path.name,
        "cache_file": cache_path.name,
    }

    if summary["nonzero_entries"] != stored_nnz:
        raise AssertionError(
            f"{cache_path.name}: cached nnz metadata does not match X."
        )
    if summary["nonzero_entries"] < 1:
        raise AssertionError(
            f"{cache_path.name}: cached matrix has no nonzero entries."
        )

    return summary, pd.Index(var.index.astype(str), dtype="object")


def build_sample_h5ad(
    source_path: Path,
    sample_id: str,
    cache_path: Path,
) -> tuple[dict, pd.Index]:
    print(f"Building sample cache: {sample_id}")

    (
        expected_cell_barcodes,
        parser_layout,
    ) = probe_historical_pandas_parser(source_path)

    print(
        "  raw layout:",
        parser_layout["layout"],
        f"(header={parser_layout['header_field_count']}, "
        f"first data row={parser_layout['first_data_field_count']})",
    )
    print(
        "  historical pandas cell columns:",
        f"{len(expected_cell_barcodes):,}",
    )

    partial_path = cache_path.with_name(
        f"{cache_path.name}.partial"
    )
    if partial_path.exists():
        partial_path.unlink()

    sparse_blocks = []
    gene_names = []
    seen_genes = set()
    total_counts = 0
    maximum_count = 0

    # This is the original parser configuration, with chunksize added.
    # It preserves pandas' implicit-index handling for these GEO files
    # without loading the entire table into RAM.
    reader = pd.read_csv(
        source_path,
        sep="\t",
        compression="gzip",
        index_col=0,
        low_memory=False,
        chunksize=GENE_CHUNK_ROWS,
    )

    try:
        for chunk_number, frame in enumerate(reader, start=1):
            observed_cell_barcodes = pd.Index(
                frame.columns.astype(str),
                name="cell_barcode",
                dtype="object",
            )

            if not observed_cell_barcodes.equals(
                expected_cell_barcodes
            ):
                raise AssertionError(
                    f"{source_path.name}: parsed cell columns changed in "
                    f"chunk {chunk_number}. Expected "
                    f"{len(expected_cell_barcodes)} columns, observed "
                    f"{len(observed_cell_barcodes)}."
                )

            chunk_genes = pd.Index(
                frame.index.astype(str),
                name="gene",
                dtype="object",
            )

            if not chunk_genes.is_unique:
                examples = (
                    chunk_genes[chunk_genes.duplicated(keep=False)]
                    .unique()
                    .tolist()[:10]
                )
                raise ValueError(
                    f"{source_path.name}: duplicate genes within chunk "
                    f"{chunk_number}, including {examples}."
                )

            repeated = [
                gene
                for gene in chunk_genes
                if gene in seen_genes
            ][:10]
            if repeated:
                raise ValueError(
                    f"{source_path.name}: gene rows repeat across chunks, "
                    f"including {repeated}."
                )

            seen_genes.update(chunk_genes)
            gene_names.extend(chunk_genes.tolist())

            try:
                values = frame.to_numpy(
                    dtype=np.float64,
                    copy=False,
                )
            except Exception as exc:
                raise ValueError(
                    f"{source_path.name}: a non-numeric count was found "
                    f"in parser chunk {chunk_number}."
                ) from exc

            integer_values, chunk_total, chunk_maximum = (
                validate_dense_count_chunk(
                    values,
                    source_path,
                    chunk_number,
                )
            )

            block = sp.csr_matrix(
                integer_values.T,
                dtype=np.int32,
            )
            block.sort_indices()
            sparse_blocks.append(block)

            total_counts += chunk_total
            maximum_count = max(maximum_count, chunk_maximum)

            if chunk_number == 1 or chunk_number % 25 == 0:
                print(
                    f"  chunk {chunk_number:03d}: "
                    f"{len(gene_names):,} genes parsed"
                )

            del frame, values, integer_values, block
            if chunk_number % 16 == 0:
                gc.collect()

    finally:
        reader.close()

    if not sparse_blocks:
        raise ValueError(
            f"{source_path.name}: no count chunks were parsed."
        )

    sample_X = sp.hstack(
        sparse_blocks,
        format="csr",
    ).astype(np.int32, copy=False)

    sample_X.sum_duplicates()
    sample_X.eliminate_zeros()
    sample_X.sort_indices()

    del sparse_blocks
    gc.collect()

    expected_shape = (
        len(expected_cell_barcodes),
        len(gene_names),
    )

    if sample_X.shape != expected_shape:
        raise AssertionError(
            f"{source_path.name}: expected sample shape "
            f"{expected_shape}, observed {sample_X.shape}."
        )
    if sample_X.nnz == 0:
        raise ValueError(
            f"{source_path.name}: parsed matrix has no nonzero values."
        )

    observed_total = int(sample_X.sum(dtype=np.int64))
    observed_maximum = int(sample_X.data.max())

    if observed_total != total_counts:
        raise AssertionError(
            f"{source_path.name}: total-count mismatch after sparse "
            f"assembly ({observed_total} != {total_counts})."
        )
    if observed_maximum != maximum_count:
        raise AssertionError(
            f"{source_path.name}: maximum-count mismatch after sparse "
            f"assembly ({observed_maximum} != {maximum_count})."
        )

    cell_ids = pd.Index(
        [
            f"{barcode}-{sample_id}"
            for barcode in expected_cell_barcodes
        ],
        name="cell_id",
        dtype="object",
    )

    obs = pd.DataFrame(index=cell_ids)
    obs["cell_barcode"] = expected_cell_barcodes.to_numpy(
        dtype=object,
        copy=True,
    )
    obs["sample_id"] = np.full(
        len(obs),
        sample_id,
        dtype=object,
    )
    obs["sample"] = np.full(
        len(obs),
        sample_id,
        dtype=object,
    )
    obs["geo_id"] = np.full(
        len(obs),
        "GSE278450",
        dtype=object,
    )
    obs["core_dataset"] = np.full(
        len(obs),
        "DS4_GSE278450",
        dtype=object,
    )
    obs["source_file"] = np.full(
        len(obs),
        source_path.name,
        dtype=object,
    )

    var = pd.DataFrame(
        index=pd.Index(
            gene_names,
            name="gene",
            dtype="object",
        )
    )

    source_stat = source_path.stat()

    sample = ad.AnnData(
        X=sample_X,
        obs=obs,
        var=var,
    )
    sample.uns["cache_version"] = CACHE_VERSION
    sample.uns["sample_id"] = sample_id
    sample.uns["source_file"] = source_path.name
    sample.uns["source_geo_accession"] = sample_id.split("_", 1)[0]
    sample.uns["source_file_size_bytes"] = int(source_stat.st_size)
    sample.uns["source_file_mtime_ns"] = int(source_stat.st_mtime_ns)
    sample.uns["header_parse_mode"] = (
        "historical pandas parser semantics with chunked reading"
    )
    sample.uns["raw_tsv_layout"] = parser_layout["layout"]
    sample.uns["raw_header_field_count"] = int(
        parser_layout["header_field_count"]
    )
    sample.uns["raw_first_data_field_count"] = int(
        parser_layout["first_data_field_count"]
    )
    sample.uns["parsed_cell_count"] = int(
        parser_layout["parsed_cell_count"]
    )
    sample.uns["parsed_index_name"] = (
        ""
        if parser_layout["parsed_index_name"] is None
        else parser_layout["parsed_index_name"]
    )
    sample.uns["gene_chunk_rows"] = int(GENE_CHUNK_ROWS)
    sample.uns["total_counts"] = int(total_counts)
    sample.uns["maximum_count"] = int(maximum_count)
    sample.uns["nonzero_entries"] = int(sample_X.nnz)

    sample.write_h5ad(
        partial_path,
        compression="lzf",
    )

    del sample, sample_X, obs, var
    gc.collect()

    summary, cached_var_names = validate_cached_sample(
        partial_path,
        source_path,
        sample_id,
    )

    os.replace(partial_path, cache_path)

    print(
        f"  saved {cache_path.name}: "
        f"{summary['cells']:,} cells × {summary['genes']:,} genes; "
        f"{summary['nonzero_entries']:,} nonzeros"
    )

    return summary, cached_var_names


In [5]:
sample_paths = []
sample_summary_rows = []
global_gene_set = set()
total_cached_cells = 0

for index, record in enumerate(input_records, start=1):
    sample_id = record["sample_id"]
    source_path = record["path"]
    cache_path = SAMPLE_CACHE_DIR / f"{index:02d}_{sample_id}.h5ad"

    print()
    print(
        f"[{index}/{len(input_records)}] "
        f"{source_path.name}"
    )

    try:
        summary, sample_var_names = validate_cached_sample(
            cache_path,
            source_path,
            sample_id,
        )
        print(
            f"  reusing validated cache: {cache_path.name}"
        )
    except Exception as cache_error:
        if cache_path.exists():
            print(
                "  cached sample is invalid and will be rebuilt:",
                repr(cache_error),
            )
            cache_path.unlink()

        summary, sample_var_names = build_sample_h5ad(
            source_path,
            sample_id,
            cache_path,
        )

    sample_paths.append(cache_path)
    sample_summary_rows.append(summary)
    total_cached_cells += int(summary["cells"])
    global_gene_set.update(sample_var_names)

    del sample_var_names
    gc.collect()


sample_summary = pd.DataFrame(sample_summary_rows)

sample_summary.to_csv(
    AUDIT_DIR / "GSE278450_raw_build_sample_summary.tsv",
    sep="\t",
    index=False,
)

observed_premerge_shape = (
    total_cached_cells,
    len(global_gene_set),
)

if len(sample_paths) != EXPECTED_N_SAMPLES:
    raise AssertionError(
        f"Expected {EXPECTED_N_SAMPLES} sample caches, "
        f"observed {len(sample_paths)}."
    )

if observed_premerge_shape != EXPECTED_OUTPUT_SHAPE:
    cell_difference = (
        EXPECTED_OUTPUT_SHAPE[0] - observed_premerge_shape[0]
    )
    gene_difference = (
        EXPECTED_OUTPUT_SHAPE[1] - observed_premerge_shape[1]
    )

    raise AssertionError(
        "The sample caches do not reproduce the historical DS4 dimensions. "
        f"Expected {EXPECTED_OUTPUT_SHAPE}, observed "
        f"{observed_premerge_shape}; cell difference={cell_difference}, "
        f"gene difference={gene_difference}. Inspect the sample summary "
        "and input inventory before concatenation."
    )

print()
print(sample_summary)
print("Pre-merge dimensions validated:", observed_premerge_shape)
print(
    "Total cached H5AD size:",
    f"{sum(path.stat().st_size for path in sample_paths) / 1024**3:.2f} GiB",
)



[1/22] GSM8546833_BT356_RawData.tsv.gz
Building sample cache: GSM8546833_BT356
  raw layout: cell-only header with unlabeled gene-index field (header=7288, first data row=7289)
  historical pandas cell columns: 7,288
  chunk 001: 128 genes parsed
  chunk 025: 3,200 genes parsed
  chunk 050: 6,400 genes parsed
  chunk 075: 9,600 genes parsed
  chunk 100: 12,800 genes parsed
  chunk 125: 16,000 genes parsed
  chunk 150: 19,200 genes parsed
  chunk 175: 22,400 genes parsed
  chunk 200: 25,600 genes parsed
  chunk 225: 28,800 genes parsed
  chunk 250: 32,000 genes parsed
  chunk 275: 35,200 genes parsed
  saved 01_GSM8546833_BT356.h5ad: 7,288 cells × 36,116 genes; 17,159,231 nonzeros

[2/22] GSM8546834_BT363_RawData.tsv.gz
Building sample cache: GSM8546834_BT363
  raw layout: cell-only header with unlabeled gene-index field (header=8986, first data row=8987)
  historical pandas cell columns: 8,986
  chunk 001: 128 genes parsed
  chunk 025: 3,200 genes parsed
  chunk 050: 6,400 genes parse

In [6]:
temporary_output = OUTPUT_H5AD.with_name(
    f"{OUTPUT_H5AD.stem}.temporary{OUTPUT_H5AD.suffix}"
)
metadata_sidecar = OUTPUT_H5AD.with_name(
    f"{OUTPUT_H5AD.stem}.uns_metadata{OUTPUT_H5AD.suffix}"
)

for stale_path in (temporary_output, metadata_sidecar):
    if stale_path.exists():
        stale_path.unlink()

cache_size_bytes = sum(path.stat().st_size for path in sample_paths)
free_bytes = shutil.disk_usage(OUTPUT_H5AD.parent).free
required_additional_bytes = max(
    10 * 1024**3,
    int(cache_size_bytes * 1.35),
)

print(
    "Free disk space before final merge:",
    f"{free_bytes / 1024**3:.2f} GiB",
)
print(
    "Additional free-space guard:",
    f"{required_additional_bytes / 1024**3:.2f} GiB",
)

if free_bytes < required_additional_bytes:
    raise OSError(
        "Insufficient disk space for the final on-disk outer join. "
        "The validated sample caches have been kept and can be reused "
        "after space is freed."
    )

build_timestamp = datetime.now().isoformat(timespec="seconds")

output_uns = {
    "dataset_id": "DS4_GSE278450",
    "geo_accession": "GSE278450",
    "build_timestamp": build_timestamp,
    "source_sample_count": int(EXPECTED_N_SAMPLES),
    "feature_join": "outer",
    "input_format": (
        "gzip-compressed gene-by-cell RawData TSV tables"
    ),
    "build_provenance": {
        "pipeline_step": "00a_ds4_build_raw_anndata",
        "memory_safe_strategy": (
            "chunked TSV parsing; per-sample sparse H5AD caches; "
            "anndata.experimental.concat_on_disk"
        ),
        "cache_version": CACHE_VERSION,
        "gene_chunk_rows": int(GENE_CHUNK_ROWS),
        "concat_max_loaded_elems": int(CONCAT_MAX_LOADED_ELEMS),
        "n_samples": int(EXPECTED_N_SAMPLES),
        "n_cells": int(EXPECTED_OUTPUT_SHAPE[0]),
        "n_genes": int(EXPECTED_OUTPUT_SHAPE[1]),
        "matrix_dtype": "int32",
        "matrix_format": "csr_matrix",
        "total_counts": int(sample_summary["total_counts"].sum()),
        "maximum_count": int(sample_summary["maximum_count"].max()),
        "source_sample_ids": np.asarray(
            [record["sample_id"] for record in input_records],
            dtype=str,
        ),
        "output_h5ad": str(OUTPUT_H5AD),
    },
}

concat_signature = inspect.signature(concat_on_disk)
concat_supported = set(concat_signature.parameters)

required_parameters = {"axis", "join"}
missing_required = sorted(required_parameters - concat_supported)
if missing_required:
    raise RuntimeError(
        "The installed anndata.concat_on_disk API is incompatible. "
        f"Missing parameters: {missing_required}; observed signature: "
        f"{concat_signature}"
    )

concat_kwargs = {
    "axis": 0,
    "join": "outer",
    "merge": "same",
    "uns_merge": None,
    "index_unique": None,
    "fill_value": 0,
    "pairwise": False,
    "max_loaded_elems": CONCAT_MAX_LOADED_ELEMS,
}
concat_kwargs = {
    key: value
    for key, value in concat_kwargs.items()
    if key in concat_supported
}

print("Starting on-disk outer join...")
print("concat_on_disk parameters:", concat_kwargs)

concat_on_disk(
    [str(path) for path in sample_paths],
    str(temporary_output),
    **concat_kwargs,
)

if not temporary_output.exists():
    raise FileNotFoundError(
        f"concat_on_disk did not create {temporary_output}"
    )

metadata_holder = ad.AnnData(
    uns=deepcopy(output_uns)
)
metadata_holder.write_h5ad(
    metadata_sidecar,
    compression="lzf",
)
del metadata_holder
gc.collect()

try:
    with h5py.File(metadata_sidecar, "r") as source_handle:
        if "uns" not in source_handle:
            raise AssertionError(
                "The metadata sidecar does not contain /uns."
            )

        with h5py.File(temporary_output, "r+") as output_handle:
            if "uns" in output_handle:
                del output_handle["uns"]

            source_handle.copy(
                "uns",
                output_handle,
                name="uns",
            )
            output_handle.flush()

finally:
    if metadata_sidecar.exists():
        metadata_sidecar.unlink()

print("Temporary merged object assembled:", temporary_output)


Free disk space before final merge: 827.06 GiB
Additional free-space guard: 10.00 GiB
Starting on-disk outer join...
concat_on_disk parameters: {'axis': 0, 'join': 'outer', 'merge': 'same', 'uns_merge': None, 'index_unique': None, 'fill_value': 0, 'pairwise': False, 'max_loaded_elems': 5000000}
Temporary merged object assembled: G:\Repository\glioma-cnv-single-cell-atlas\data\interim\DS4_GSE278450\GSE278450_merged_rawcounts.temporary.h5ad


In [7]:
def validate_sparse_csr_on_disk(
    group: h5py.Group,
    expected_shape: tuple[int, int],
    data_chunk_size: int,
) -> dict:
    encoding = stored_sparse_encoding(group)
    if encoding != "csr_matrix":
        raise AssertionError(
            f"Expected CSR matrix encoding, observed {encoding!r}."
        )

    shape = stored_matrix_shape(group)
    if shape != expected_shape:
        raise AssertionError(
            f"Expected matrix shape {expected_shape}, observed {shape}."
        )

    for key in ("data", "indices", "indptr"):
        if key not in group:
            raise AssertionError(
                f"Sparse CSR group is missing dataset {key!r}."
            )

    data = group["data"]
    indices = group["indices"]
    indptr = group["indptr"]

    if data.dtype != np.dtype(np.int32):
        raise AssertionError(
            f"Expected int32 count data, observed {data.dtype}."
        )
    if len(indptr) != expected_shape[0] + 1:
        raise AssertionError(
            "CSR indptr has an unexpected length."
        )
    if int(indptr[0]) != 0:
        raise AssertionError("CSR indptr must begin at zero.")
    if int(indptr[-1]) != len(data):
        raise AssertionError(
            "CSR indptr endpoint does not equal the number of nonzeros."
        )
    if len(indices) != len(data):
        raise AssertionError(
            "CSR data and indices lengths differ."
        )

    previous_pointer = 0
    for start in range(0, len(indptr), data_chunk_size):
        stop = min(start + data_chunk_size, len(indptr))
        pointer_chunk = np.asarray(
            indptr[start:stop],
            dtype=np.int64,
        )

        if start > 0:
            pointer_chunk = np.concatenate(
                [
                    np.asarray([previous_pointer], dtype=np.int64),
                    pointer_chunk,
                ]
            )

        if np.any(np.diff(pointer_chunk) < 0):
            raise AssertionError(
                "CSR indptr is not monotonically nondecreasing."
            )

        previous_pointer = int(pointer_chunk[-1])

    total_counts = 0
    maximum_count = 0

    for start in range(0, len(data), data_chunk_size):
        stop = min(start + data_chunk_size, len(data))

        data_chunk = np.asarray(
            data[start:stop],
            dtype=np.int32,
        )
        index_chunk = np.asarray(
            indices[start:stop],
            dtype=np.int64,
        )

        if data_chunk.size:
            if np.any(data_chunk <= 0):
                raise AssertionError(
                    "Stored sparse data contains zero or negative values."
                )
            maximum_count = max(
                maximum_count,
                int(data_chunk.max()),
            )
            total_counts += int(
                data_chunk.sum(dtype=np.int64)
            )

        if index_chunk.size:
            if index_chunk.min() < 0:
                raise AssertionError(
                    "CSR indices contain a negative column index."
                )
            if index_chunk.max() >= expected_shape[1]:
                raise AssertionError(
                    "CSR indices contain an out-of-range column index."
                )

    return {
        "shape": shape,
        "encoding": encoding,
        "dtype": str(data.dtype),
        "nonzero_entries": int(len(data)),
        "total_counts": int(total_counts),
        "maximum_count": int(maximum_count),
    }


try:
    with h5py.File(temporary_output, "r") as check_handle:
        for key in ("X", "obs", "var", "uns"):
            if key not in check_handle:
                raise AssertionError(
                    f"Saved output is missing H5AD group /{key}."
                )

        matrix_summary = validate_sparse_csr_on_disk(
            check_handle["X"],
            EXPECTED_OUTPUT_SHAPE,
            VALIDATION_DATA_CHUNK,
        )

        saved_obs = read_elem(check_handle["obs"])
        saved_var = read_elem(check_handle["var"])
        saved_uns = read_elem(check_handle["uns"])

    if len(saved_obs) != EXPECTED_OUTPUT_SHAPE[0]:
        raise AssertionError(
            "Saved obs table has an unexpected number of rows."
        )
    if len(saved_var) != EXPECTED_OUTPUT_SHAPE[1]:
        raise AssertionError(
            "Saved var table has an unexpected number of rows."
        )
    if not saved_obs.index.is_unique:
        raise AssertionError(
            "Saved cell identifiers are not unique."
        )
    if not saved_var.index.is_unique:
        raise AssertionError(
            "Saved gene identifiers are not unique."
        )

    required_obs_columns = (
        "cell_barcode",
        "sample_id",
        "sample",
        "geo_id",
        "core_dataset",
        "source_file",
    )
    missing_obs_columns = [
        column
        for column in required_obs_columns
        if column not in saved_obs.columns
    ]
    if missing_obs_columns:
        raise AssertionError(
            "Saved output is missing obs columns: "
            f"{missing_obs_columns}"
        )

    observed_sample_ids = set(
        saved_obs["sample_id"].astype(str).unique()
    )
    if observed_sample_ids != EXPECTED_SAMPLE_IDS:
        raise AssertionError(
            "Saved output contains an unexpected sample-ID set."
        )

    observed_counts = (
        saved_obs["sample_id"]
        .astype(str)
        .value_counts()
        .sort_index()
    )
    expected_counts = (
        sample_summary
        .set_index("sample_id")["cells"]
        .astype(np.int64)
        .sort_index()
    )
    pd.testing.assert_series_equal(
        observed_counts.astype(np.int64),
        expected_counts,
        check_names=False,
    )

    required_uns_keys = (
        "dataset_id",
        "geo_accession",
        "build_timestamp",
        "source_sample_count",
        "feature_join",
        "input_format",
        "build_provenance",
    )
    missing_uns_keys = [
        key for key in required_uns_keys
        if key not in saved_uns
    ]
    if missing_uns_keys:
        raise AssertionError(
            "Saved output is missing uns entries: "
            f"{missing_uns_keys}"
        )

    expected_total_counts = int(
        sample_summary["total_counts"].sum()
    )
    expected_maximum_count = int(
        sample_summary["maximum_count"].max()
    )

    if matrix_summary["total_counts"] != expected_total_counts:
        raise AssertionError(
            "Final total counts do not match the sum of sample caches: "
            f"{matrix_summary['total_counts']} != "
            f"{expected_total_counts}"
        )
    if matrix_summary["maximum_count"] != expected_maximum_count:
        raise AssertionError(
            "Final maximum count does not match the sample caches: "
            f"{matrix_summary['maximum_count']} != "
            f"{expected_maximum_count}"
        )

    os.replace(temporary_output, OUTPUT_H5AD)

except Exception:
    print(
        "Final validation failed. The temporary merged output and "
        "validated sample caches were retained for diagnosis."
    )
    raise


build_summary = {
    "dataset_id": "DS4_GSE278450",
    "geo_accession": "GSE278450",
    "n_samples": int(EXPECTED_N_SAMPLES),
    "n_cells": int(EXPECTED_OUTPUT_SHAPE[0]),
    "n_genes": int(EXPECTED_OUTPUT_SHAPE[1]),
    "matrix_format": matrix_summary["encoding"],
    "matrix_dtype": matrix_summary["dtype"],
    "nonzero_entries": matrix_summary["nonzero_entries"],
    "total_counts": matrix_summary["total_counts"],
    "maximum_count": matrix_summary["maximum_count"],
    "output_h5ad": str(OUTPUT_H5AD),
    "output_size_bytes": int(OUTPUT_H5AD.stat().st_size),
    "build_timestamp": build_timestamp,
}

with open(
    AUDIT_DIR / "GSE278450_raw_build_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(build_summary, handle, indent=2)

if SAMPLE_CACHE_DIR.exists():
    shutil.rmtree(SAMPLE_CACHE_DIR)

gc.collect()

print()
print("Saved and validated:", OUTPUT_H5AD)
print("Validated shape:", EXPECTED_OUTPUT_SHAPE)
print("Validated X encoding: csr_matrix")
print("Validated X dtype: int32")
print("Validated samples:", EXPECTED_N_SAMPLES)
print(
    "Validated nonzero entries:",
    f"{matrix_summary['nonzero_entries']:,}",
)
print(build_summary)



Saved and validated: G:\Repository\glioma-cnv-single-cell-atlas\data\interim\DS4_GSE278450\GSE278450_merged_rawcounts.h5ad
Validated shape: (91336, 36116)
Validated X encoding: csr_matrix
Validated X dtype: int32
Validated samples: 22
Validated nonzero entries: 187,038,712
{'dataset_id': 'DS4_GSE278450', 'geo_accession': 'GSE278450', 'n_samples': 22, 'n_cells': 91336, 'n_genes': 36116, 'matrix_format': 'csr_matrix', 'matrix_dtype': 'int32', 'nonzero_entries': 187038712, 'total_counts': 558528497, 'maximum_count': 17267, 'output_h5ad': 'G:\\Repository\\glioma-cnv-single-cell-atlas\\data\\interim\\DS4_GSE278450\\GSE278450_merged_rawcounts.h5ad', 'output_size_bytes': 1512868156, 'build_timestamp': '2026-08-05T10:40:34'}
